# Sentinel-2 optical baseline

Sentinel-2 Random Forest evaluated against CROME 2022 holdout  labels.

## setup

In [ ]:
#Install dependencies
RUN_INSTALLS = False  
if RUN_INSTALLS:
    %pip install -q earthengine-api geemap geopandas pyogrio shapely pandas numpy scikit-learn matplotlib seaborn tqdm

## Configuration

In [ ]:
#Configure paths, classes, and output folders
from pathlib import Path
from datetime import datetime
import json
import warnings
#Load Earth Engine dependencies.
import ee
import geemap
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

BASELINE_ROOT = Path.cwd()
DATA_RAW = BASELINE_ROOT / 'data' / 'raw'
DATA_PROCESSED = BASELINE_ROOT / 'processed_data'
OUTPUTS = BASELINE_ROOT / 'outputs'
TABLES = OUTPUTS / 'tables'
FIGURES = OUTPUTS / 'figures'
LOGS = OUTPUTS / 'logs'
for folder in [DATA_RAW, DATA_PROCESSED, TABLES, FIGURES, LOGS]:
    folder.mkdir(parents=True, exist_ok=True)
CROME_DIR = DATA_RAW / 'crome' / 'crome_2022_refined' / 'east_anglia'
CROME_FILES = [
    'Crop_Map_of_England_2022_Suffolk_refined.geojson',
    'Crop_Map_of_England_2022_Cambridgeshire_refined.geojson',
    'Crop_Map_of_England_2022_Norfolk_refined.geojson',
]
CROME_LOOKUP_CSV = DATA_RAW / 'crome_crop_lookup_refined.csv'
UKCEH_2022_GPKG = DATA_RAW / 'ukceh' / 'ukceh_east_anglia' / 'lccm-2022_6395063' / 'lccm-2022_6395063.gpkg'
EE_PROJECT_ID = 'dissertation-496916'
S2_EXPORT_DRIVE_FOLDER = 'dissertation_baseline_exports'
S2_EXPORT_DESCRIPTION = 's2_2022_east_anglia_crome_8class_baseline'
S2_SAMPLE_CSV = DATA_PROCESSED / 's2_2022_east_anglia_crome_8class_baseline.csv'
RANDOM_SEED = 42
DEBUG_N_PER_CLASS = 300
BASELINE_N_PER_CLASS = 1000
TEST_SIZE = 0.30
print('Notebook root:', BASELINE_ROOT)
print('Raw data root:', DATA_RAW)

In [ ]:
#Define the eight classes harmonisation
MAIN_CLASSES = [
    {'class_id': 1, 'label': 'Winter wheat', 'source_classes': ['Winter wheat']},
    {'class_id': 2, 'label': 'Winter barley', 'source_classes': ['Winter barley']},
    {'class_id': 3, 'label': 'Spring barley', 'source_classes': ['Spring barley']},
    {'class_id': 4, 'label': 'Beet (sugar beet / fodder beet)', 'source_classes': ['Beet (sugar beet / fodder beet)', 'Beet']},
    {'class_id': 5, 'label': 'Maize', 'source_classes': ['Maize']},
    {'class_id': 6, 'label': 'Oilseed rape', 'source_classes': ['Oilseed rape', 'Winter Oilseed']},
    {'class_id': 7, 'label': 'Potatoes', 'source_classes': ['Potatoes', 'Potato']},
    {'class_id': 8, 'label': 'Pulses / field beans and peas', 'source_classes': ['Spring field beans', 'Winter field beans', 'Peas']},
]
CLASS_LABELS = [item['label'] for item in MAIN_CLASSES]
SOURCE_TO_MAIN = {}
for item in MAIN_CLASSES:
    for source_class in item['source_classes']:
        SOURCE_TO_MAIN[source_class] = (item['class_id'], item['label'])
retained = []
for item in MAIN_CLASSES:
    retained.append({
        'class_id': item['class_id'],
        'analysis_class': item['label'],
        'source_classes': '; '.join(item['source_classes']),
        'decision': 'aggregate_main' if item['label'] == 'Pulses / field beans and peas' else 'main',
    })

## Imports and utility functions

In [ ]:
#define utility functions.
try:
    import geopandas as gpd
except ImportError as exc:
    raise ImportError('geopandas is missing. Set RUN_INSTALLS=True in the install cell and rerun it.') from exc
def add_main_class_columns(df, source_column='analysis_class'):
    out = df.copy()
    out['source_analysis_class'] = out[source_column]
    out['class_id'] = out[source_column].map(lambda value: SOURCE_TO_MAIN.get(value, (pd.NA, pd.NA))[0])
    out['analysis_class'] = out[source_column].map(lambda value: SOURCE_TO_MAIN.get(value, (pd.NA, pd.NA))[1])
    return out.dropna(subset=['class_id', 'analysis_class'])
def balanced_sample(gdf, n_per_class, seed):
    return (
        gdf.groupby('analysis_class', group_keys=False)
        .apply(lambda group: group.sample(n=min(len(group), n_per_class), random_state=seed))
        .reset_index(drop=True)
    )
def representative_points(polygons):
    points = polygons.copy()
    points['geometry'] = points.geometry.representative_point()
    return points.to_crs('EPSG:4326')
def clean_gpkg_columns(gdf):
    out = gdf.copy()
    geom_name = out.geometry.name
    new_columns = []
    seen = set()
    for col in list(out.columns):
        if col == geom_name:
            safe = geom_name
        else:
            safe = str(col).strip().replace(' ', '_').replace('/', '_').replace('-', '_')
            safe = safe[:55] or 'field'
        base = safe
        i = 1
        while safe.lower() in seen:
            suffix = f'_{i}'
            safe = f'{base[:55-len(suffix)]}{suffix}'
            i += 1
        seen.add(safe.lower())
        new_columns.append(safe)
    out.columns = new_columns
    if 'LUCODE' in out.columns and 'lucode' in out.columns:
        out = out.drop(columns=['LUCODE'])
    if 'LUCODE_1' in out.columns and 'lucode' in out.columns:
        out = out.drop(columns=['LUCODE_1'])
    if not out.columns.is_unique:
        out = out.loc[:, ~out.columns.duplicated()]
    return gpd.GeoDataFrame(out, geometry=geom_name, crs=gdf.crs)
pd.DataFrame(retained).to_csv(TABLES / 'retained_8_main_classes.csv', index=False)
display(pd.DataFrame(retained))

## Prepare CROME samples

In [ ]:
#Prepare stratified CROME sample points.
RUN_PREPARE_SAMPLES = True
WRITE_FULL_8CLASS_GPKG = True
if RUN_PREPARE_SAMPLES:
    missing = [str(CROME_DIR / name) for name in CROME_FILES if not (CROME_DIR / name).exists()]
    if missing:
        raise FileNotFoundError('Missing CROME file(s):\n' + '\n'.join(missing))
    if not CROME_LOOKUP_CSV.exists():
        raise FileNotFoundError(f'Missing lookup CSV: {CROME_LOOKUP_CSV}')
    lookup = pd.read_csv(CROME_LOOKUP_CSV)
    frames = []
    raw_count_rows = []
    for file_name in tqdm(CROME_FILES, desc='Reading CROME county files'):
        file_path = CROME_DIR / file_name
        gdf = gpd.read_file(file_path, engine='pyogrio')
        if gdf.crs is None:
            gdf = gdf.set_crs('EPSG:27700')
        if 'analysis_class' not in gdf.columns:
            join_col = 'lucode' if 'lucode' in gdf.columns else 'Lucode'
            gdf = gdf.merge(lookup, left_on=join_col, right_on='lucode', how='left')
        raw_counts = gdf['analysis_class'].value_counts(dropna=False).rename_axis('source_analysis_class').reset_index(name='count')
        raw_counts['source_file'] = file_name
        raw_count_rows.append(raw_counts)
        gdf = add_main_class_columns(gdf, source_column='analysis_class')
        gdf['county_source_file'] = file_name
        gdf['sample_uid'] = gdf['county_source_file'].str.replace('.geojson', '', regex=False) + '_' + gdf.index.astype(str)
        keep_cols = [
            'sample_uid', 'county_source_file', 'cromeid', 'lucode',
            'crome_original_name', 'ukceh_aligned_name', 'source_analysis_class',
            'analysis_class', 'class_id', 'prob', 'shape_area', 'geometry'
        ]
        keep_cols = [col for col in keep_cols if col in gdf.columns]
        frames.append(gdf[keep_cols])
    crome_8 = pd.concat(frames, ignore_index=True)
    crome_8['class_id'] = crome_8['class_id'].astype(int)
    harmonised_counts = pd.concat(raw_count_rows, ignore_index=True)
    harmonised_counts = harmonised_counts.groupby('source_analysis_class', as_index=False)['count'].sum().sort_values('count', ascending=False)
    harmonised_counts.to_csv(TABLES / 'harmonised_class_count_table.csv', index=False)
    crome_counts = crome_8.groupby(['class_id', 'analysis_class'], as_index=False).size().rename(columns={'size': 'count'})
    crome_counts.to_csv(TABLES / 'crome_8class_counts.csv', index=False)
    debug_polygons = balanced_sample(crome_8, DEBUG_N_PER_CLASS, RANDOM_SEED)
    baseline_polygons = balanced_sample(crome_8, BASELINE_N_PER_CLASS, RANDOM_SEED)
    crome_8 = clean_gpkg_columns(crome_8)
    debug_points = clean_gpkg_columns(representative_points(debug_polygons))
    baseline_points = clean_gpkg_columns(representative_points(baseline_polygons))
    debug_points_path = DATA_PROCESSED / 'crome_2022_east_anglia_8class_points_debug.gpkg'
    baseline_points_path = DATA_PROCESSED / 'crome_2022_east_anglia_8class_points_baseline.gpkg'
    debug_points.to_file(debug_points_path, layer='points_debug', driver='GPKG')
    baseline_points.to_file(baseline_points_path, layer='points_baseline', driver='GPKG')
    if WRITE_FULL_8CLASS_GPKG:
        crome_8.to_file(DATA_PROCESSED / 'crome_2022_east_anglia_8class.gpkg', layer='crome_8class', driver='GPKG')
    sampling_plan = pd.DataFrame([
        {'sample_name': 'debug', 'n_per_class_target': DEBUG_N_PER_CLASS, 'n_total': len(debug_points), 'method': 'class-balanced random representative points', 'random_seed': RANDOM_SEED},
        {'sample_name': 'baseline', 'n_per_class_target': BASELINE_N_PER_CLASS, 'n_total': len(baseline_points), 'method': 'class-balanced random representative points', 'random_seed': RANDOM_SEED},
    ])
    sampling_plan.to_csv(TABLES / 'sampling_plan.csv', index=False)
    display(crome_counts)
    display(sampling_plan)
    print('Wrote:', debug_points_path)
    print('Wrote:', baseline_points_path)

## Export Sentinel-2 features

In [ ]:
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
S2_INDICES = ['NDVI', 'NDRE', 'LSWI', 'EVI']
SEASONAL_WINDOWS = {
    'winter_establishment': ('2021-10-20', '2022-02-28'),
    'spring_growth': ('2022-03-01', '2022-05-31'),
    'summer_peak': ('2022-06-01', '2022-08-31'),
    'late_season': ('2022-09-01', '2022-09-30'),
}
def initialize_ee(project_id):
    try:
        ee.Initialize(project=project_id)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project_id)
def mask_s2_sr_harmonized(image):
    scl = image.select('SCL')
    clear = scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(clear).divide(10000).copyProperties(image, ['system:time_start'])
def add_s2_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndre = image.normalizedDifference(['B8A', 'B5']).rename('NDRE')
    lswi = image.normalizedDifference(['B8', 'B11']).rename('LSWI')
    evi = image.expression(
        '2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))',
        {'nir': image.select('B8'), 'red': image.select('B4'), 'blue': image.select('B2')},
    ).rename('EVI')
    return image.addBands([ndvi, ndre, lswi, evi])
def seasonal_s2_composite(aoi, start, end, season_name):
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 70))
        .select(S2_BANDS + ['SCL'])
        .map(mask_s2_sr_harmonized)
        .map(add_s2_indices)
    )
    predictors = S2_BANDS + S2_INDICES
    renamed = [f'{season_name}_{band}' for band in predictors]
    return collection.select(predictors).median().rename(renamed)
def build_s2_feature_stack(aoi):
    composites = []
    for season_name, (start, end) in SEASONAL_WINDOWS.items():
        composites.append(seasonal_s2_composite(aoi, start, end, season_name))
    return ee.Image.cat(composites)
S2_FEATURE_BANDS = [f'{season}_{band}' for season in SEASONAL_WINDOWS for band in (S2_BANDS + S2_INDICES)]
pd.DataFrame({'band_name': S2_FEATURE_BANDS}).to_csv(TABLES / 's2_feature_band_list.csv', index=False)
print('Sentinel-2 feature count:', len(S2_FEATURE_BANDS))

In [ ]:
#Build and export Sentinel-2 features.
RUN_GEE_EXPORT = False  
SAMPLE_MODE = 'debug'  
if RUN_GEE_EXPORT:
    initialize_ee(EE_PROJECT_ID)
    if SAMPLE_MODE == 'debug':
        sample_path = DATA_PROCESSED / 'crome_2022_east_anglia_8class_points_debug.gpkg'
        sample_layer = 'points_debug'
    else:
        sample_path = DATA_PROCESSED / 'crome_2022_east_anglia_8class_points_baseline.gpkg'
        sample_layer = 'points_baseline'
    samples = gpd.read_file(sample_path, layer=sample_layer).to_crs('EPSG:4326')
    samples = samples[[col for col in ['sample_uid', 'class_id', 'analysis_class'] if col in samples.columns] + ['geometry']]
    sample_fc = geemap.geopandas_to_ee(samples, geodesic=False)
    aoi = sample_fc.geometry().bounds().buffer(20000)
    s2_stack = build_s2_feature_stack(aoi)
    sampled = s2_stack.sampleRegions(
        collection=sample_fc,
        properties=['sample_uid', 'class_id', 'analysis_class'],
        scale=10,
        geometries=False,
        tileScale=4,
    )
    description = f'{S2_EXPORT_DESCRIPTION}_{SAMPLE_MODE}'
    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=description,
        folder=S2_EXPORT_DRIVE_FOLDER,
        fileNamePrefix=description,
        fileFormat='CSV',
    )
    task.start()
    log_text = '\n'.join([
        f'timestamp={datetime.now().isoformat(timespec="seconds")}',
        f'sample_mode={SAMPLE_MODE}',
        f'sample_path={sample_path}',
        f'local_sample_count={len(samples)}',
        f'export_description={description}',
        f'drive_folder={S2_EXPORT_DRIVE_FOLDER}',
        f'task_id={task.id}',
        'cloud_mask=SCL clear classes excluding no-data, saturated, shadow, cloud, cirrus and snow/ice',
        'reference_note=CROME is a reference crop-map product, not direct ground truth',
    ])
    (LOGS / 's2_pipeline_log.txt').write_text(log_text, encoding='utf-8')
    print(log_text)
else:
    print('RUN_GEE_EXPORT is False. Set it to True when ready to export Sentinel-2 features.')

## Random Forest baseline

In [ ]:
#Train and evaluate the Sentinel-2 optical RF baseline.
RUN_RF_BASELINE = False  
if RUN_RF_BASELINE:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
    import matplotlib.pyplot as plt
    import seaborn as sns
    if not S2_SAMPLE_CSV.exists():
        raise FileNotFoundError(f'Missing Sentinel-2 sample CSV: {S2_SAMPLE_CSV}')
    features = pd.read_csv(S2_SAMPLE_CSV)
    feature_cols = [col for col in S2_FEATURE_BANDS if col in features.columns]
    missing_required = [col for col in ['analysis_class', 'class_id'] if col not in features.columns]
    if missing_required:
        raise ValueError(f'Missing required columns in exported sample table: {missing_required}')
    model_df = features.dropna(subset=feature_cols + ['analysis_class']).copy()
    model_df = model_df[model_df['analysis_class'].isin(CLASS_LABELS)]
    X = model_df[feature_cols]
    y = model_df['analysis_class']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_SEED,
        class_weight='balanced_subsample',
        n_jobs=-1,
        max_features='sqrt',
    )
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=CLASS_LABELS)
    cm_df = pd.DataFrame(cm, index=CLASS_LABELS, columns=CLASS_LABELS)
    cm_df.to_csv(TABLES / 's2_only_confusion_matrix.csv')
    report = classification_report(y_test, pred, labels=CLASS_LABELS, output_dict=True, zero_division=0)
    class_metrics = (
        pd.DataFrame(report).transpose().loc[CLASS_LABELS, ['precision', 'recall', 'f1-score', 'support']]
        .rename(columns={'precision': 'user_accuracy', 'recall': 'producer_accuracy'})
    )
    class_metrics.to_csv(TABLES / 's2_only_class_metrics.csv')
    overall = {
        'overall_agreement': accuracy_score(y_test, pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
        'macro_f1': f1_score(y_test, pred, labels=CLASS_LABELS, average='macro'),
        'weighted_f1': f1_score(y_test, pred, labels=CLASS_LABELS, average='weighted'),
        'validation_note': 'Random split baseline; interpret with spatial leakage risk.',
    }
    pd.DataFrame([overall]).to_csv(TABLES / 's2_only_overall_metrics.csv', index=False)
    importance = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
    importance.to_csv(TABLES / 's2_only_feature_importance.csv', index=False)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True)
    plt.xlabel('Predicted class')
    plt.ylabel('CROME reference label')
    plt.title('Sentinel-2-only Random Forest agreement with CROME')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES / 's2_only_confusion_matrix.png', dpi=300)
    plt.show()
    log_text = '\n'.join([
        f'timestamp={datetime.now().isoformat(timespec="seconds")}',
        f'sample_csv={S2_SAMPLE_CSV}',
        f'rows_used={len(model_df)}',
        f'predictor_count={len(feature_cols)}',
        f'test_size={TEST_SIZE}',
        f'random_seed={RANDOM_SEED}',
        'model=RandomForestClassifier(n_estimators=500, class_weight=balanced_subsample, max_features=sqrt)',
        'validation_note=random train-test split used for first baseline; discuss spatial leakage risk',
        'reference_note=CROME treated as a reference crop-map product rather than direct ground truth',
    ])
    (LOGS / 's2_only_model_log.txt').write_text(log_text, encoding='utf-8')
    display(cm_df)
    display(class_metrics)
    display(pd.DataFrame([overall]))
    display(importance.head(20))
else:
    print('RUN_RF_BASELINE is False. Set it to True after the Sentinel-2 feature CSV exists.')